In [361]:
import pandas as pd

df = pd.read_csv("../data/spam.csv", encoding="latin-1")

print("Shape:", df.shape)
print(df.dtypes)
print(df.head())

Shape: (5572, 5)
v1            str
v2            str
Unnamed: 2    str
Unnamed: 3    str
Unnamed: 4    str
dtype: object
     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [362]:

print(df.isnull().sum())

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64


In [363]:
df = df.drop(columns=["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"])

In [364]:
print(df.shape)
print(df.columns)
print(df.isnull().sum())

(5572, 2)
Index(['v1', 'v2'], dtype='str')
v1    0
v2    0
dtype: int64


In [365]:
df = df.rename(columns={
    "v1": "label",
    "v2": "message"
})

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [366]:
df["labeled"] = df["label"].map({
    "spam": 0,
    "ham": 1
})

df.head()

,label,message,labeled
0,ham,"Go until jurong point, crazy.. Available only ...",1
1,ham,Ok lar... Joking wif u oni...,1
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,0
3,ham,U dun say so early hor... U c already then say...,1
4,ham,"Nah I don't think he goes to usf, he lives aro...",1


In [367]:
# Number of spam and ham messages
class_counts = df["label"].value_counts()

print("Class counts:")
print(class_counts)

class_percentages = df["label"].value_counts(normalize=True) * 100

print("\nClass percentages:")
print(class_percentages.round(2))

ham_count = (df["label"] == "ham").sum()
spam_count = (df["label"] == "spam").sum()

imbalance_ratio = ham_count / spam_count

print("\nHam messages:", ham_count)
print("Spam messages:", spam_count)
print("Ham-to-spam ratio:", round(imbalance_ratio, 2))

Class counts:
label
ham     4825
spam     747
Name: count, dtype: int64

Class percentages:
label
ham     86.59
spam    13.41
Name: proportion, dtype: float64

Ham messages: 4825
Spam messages: 747
Ham-to-spam ratio: 6.46


In [368]:
# A list of contractions from http://stackoverflow.com/questions/19790188/expanding-english-language-contractions-in-python
contractions = {

"you've":"you have",
"ain't": "am not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he's": "he is",
"how'd": "how did",
"how'll": "how will",
"how's": "how is",
"i'd": "i would",
"i'll": "i will",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'll": "it will",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"must've": "must have",
"mustn't": "must not",
"needn't": "need not",
"oughtn't": "ought not",
"shan't": "shall not",
"sha'n't": "shall not",
"she'd": "she would",
"she'll": "she will",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"that'd": "that would",
"that's": "that is",
"there'd": "there had",
"there's": "there is",
"they'd": "they would",
"they'll": "they will",
"they're": "they are",
"they've": "they have",
"wasn't": "was not",
"we'd": "we would",
"we'll": "we will",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"where'd": "where did",
"where's": "where is",
"who'll": "who will",
"who's": "who is",
"won't": "will not",
"wouldn't": "would not",
"you'd": "you would",
"you'll": "you will",
"you're": "you are"
}

In [369]:
import re

# Sort longest contractions first so "can't've" is matched before "can't"
contraction_pattern = re.compile(
    r"(?<!\w)("
    + "|".join(
        re.escape(key)
        for key in sorted(contractions, key=len, reverse=True)
    )
    + r")(?!\w)"
)


def expand_contractions(text):
    return contraction_pattern.sub(
        lambda match: contractions[match.group(0)],
        text
    )

In [370]:
text = "i'm sure you can't go because you've called late"

print(expand_contractions(text))

i am sure you cannot go because you have called late


In [371]:
import re
import html
import unicodedata
import pandas as pd


def clean_message(text):
    """Clean an SMS message while preserving useful spam indicators."""

    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML characters such as &amp;
    text = html.unescape(text)

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Normalize apostrophes and convert to lowercase
    text = text.replace("’", "'").lower()
    text = expand_contractions(text)

    # Replace important patterns with tokens because these patterns could be 
    #a spam feature
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " urltoken ",
        text
    )

    text = re.sub(
        r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b",
        " emailtoken ",
        text
    )

    text = re.sub(
        r"(?:\+?\d[\d\s().-]{7,}\d)",
        " phonetoken ",
        text
    )

    text = re.sub(
        r"(?:£|\$|€)\s?\d+(?:[.,]\d+)?",
        " moneytoken ",

        text
    )

    text = re.sub(
        r"\b\d+(?:[.,]\d+)?\b",
        " numbertoken ",
        text
    )

    # Keep letters, words, apostrophes, ! and ?
    text = re.sub(r"[^\w\s!?']", " ", text)

    # Remove repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [372]:
example = (
    "I'm sure you can't visit www.test.com. "
    "Call +44 123 456 7890 to win £500!"
)

print("Original:")
print(example)

print("\nCleaned:")
print(clean_message(example))

Original:
I'm sure you can't visit www.test.com. Call +44 123 456 7890 to win £500!

Cleaned:
i am sure you cannot visit urltoken call phonetoken to win moneytoken !


In [373]:
df = df.copy()

df["clean_message"] = df["message"].apply(clean_message)

df[["message", "clean_message"]].head()

,message,clean_message
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in numbertoken a wkly comp to win f...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i do not think he goes to usf he lives aro...


In [374]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [375]:
from nltk import pos_tag
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download("averaged_perceptron_tagger_eng")
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words("english"))

# Preserve negation words because they affect meaning
stop_words.discard("no")
stop_words.discard("not")
stop_words.discard("nor")

special_tokens = {
    "urltoken",
    "emailtoken",
    "phonetoken",
    "moneytoken",
    "numbertoken"
}

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [376]:
"""def preprocess_nlp(text):
    
    Tokenize text, remove stopwords, and apply lemmatization.
    

    if pd.isna(text):
        return ""

    tokens = word_tokenize(text)

    processed_tokens = []

    for token in tokens:
        token = token.lower()

        # Keep important punctuation for spam and ham
        if token in {"!", "?"}:
            processed_tokens.append(token)
            continue

        # Keep the special spam indicator tokens
        if token in special_tokens:
            processed_tokens.append(token)
            continue

        # Ignore other punctuation
        if not token.isalpha():
            continue

        # Remove stopwords but preserve no/not/nor(the negation would help us)
        if token in stop_words:
            continue

        # Simple lemmatization without POS tagging
        lemma = lemmatizer.lemmatize(token)

        processed_tokens.append(lemma)

    return " ".join(processed_tokens)"""

'def preprocess_nlp(text):\n\n    Tokenize text, remove stopwords, and apply lemmatization.\n\n\n    if pd.isna(text):\n        return ""\n\n    tokens = word_tokenize(text)\n\n    processed_tokens = []\n\n    for token in tokens:\n        token = token.lower()\n\n        # Keep important punctuation for spam and ham\n        if token in {"!", "?"}:\n            processed_tokens.append(token)\n            continue\n\n        # Keep the special spam indicator tokens\n        if token in special_tokens:\n            processed_tokens.append(token)\n            continue\n\n        # Ignore other punctuation\n        if not token.isalpha():\n            continue\n\n        # Remove stopwords but preserve no/not/nor(the negation would help us)\n        if token in stop_words:\n            continue\n\n        # Simple lemmatization without POS tagging\n        lemma = lemmatizer.lemmatize(token)\n\n        processed_tokens.append(lemma)\n\n    return " ".join(processed_tokens)'

In [377]:
def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ

    if tag.startswith("V"):
        return wordnet.VERB

    if tag.startswith("N"):
        return wordnet.NOUN

    if tag.startswith("R"):
        return wordnet.ADV

    return wordnet.NOUN

In [378]:
def preprocess_nlp(text):
    """
    Tokenize text, remove stopwords, and apply POS-aware lemmatization.
    """

    if pd.isna(text):
        return ""

    # Split the message into tokens
    tokens = word_tokenize(str(text))

    # Determine the grammatical role of every token
    tagged_tokens = pos_tag(tokens)

    processed_tokens = []

    for token, tag in tagged_tokens:
        token = token.lower()

        # Preserve useful punctuation
        if token in {"!", "?"}:
            processed_tokens.append(token)
            continue

        # Preserve tokens created during cleaning
        if token in special_tokens:
            processed_tokens.append(token)
            continue

        # Remove other punctuation and invalid tokens
        if not token.isalpha():
            continue

        # Lemmatize using the correct grammatical role
        lemma = lemmatizer.lemmatize(
            token,
            pos=get_wordnet_pos(tag)
        )

        # Remove stopwords after lemmatization
        if token in stop_words or lemma in stop_words:
            continue

        processed_tokens.append(lemma)

    return " ".join(processed_tokens)

In [379]:
df["lemmatized_text"] = df["clean_message"].apply(preprocess_nlp)

In [380]:
df[
    ["message", "clean_message", "lemmatized_text", "label"]
].head(20)

,message,clean_message,lemmatized_text,label
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,go jurong point crazy available bugis n great ...,ham
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni,ok lar joking wif u oni,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in numbertoken a wkly comp to win f...,free entry numbertoken wkly comp win fa cup fi...,spam
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,u dun say early hor u c already say,ham
4,"Nah I don't think he goes to usf, he lives aro...",nah i do not think he goes to usf he lives aro...,nah not think go usf live around though,ham
5,FreeMsg Hey there darling it's been 3 week's n...,freemsg hey there darling it is been numbertok...,freemsg hey darling numbertoken week no word b...,spam
6,Even my brother is not like to speak with me. ...,even my brother is not like to speak with me t...,even brother not like speak treat like aid patent,ham
7,As per your request 'Melle Melle (Oru Minnamin...,as per your request 'melle melle oru minnaminu...,per request melle oru minnaminunginte nurungu ...,ham
8,WINNER!! As a valued network customer you have...,winner!! as a valued network customer you have...,winner ! ! value network customer select recei...,spam
9,Had your mobile 11 months or more? U R entitle...,had your mobile numbertoken months or more? u ...,mobile numbertoken month ? u r entitle update ...,spam


checking if the nlp preprocessing made any duplicate messages.


In [381]:
print(
    "Empty lemmatized messages:",
    (df["lemmatized_text"].str.strip() == "").sum()
)

print(
    "Duplicates after NLP preprocessing:",
    df["lemmatized_text"].duplicated().sum()
)

Empty lemmatized messages: 6
Duplicates after NLP preprocessing: 505


In [382]:
df = df[
    df["lemmatized_text"].str.strip() != ""
].copy()

In [383]:
df = df.drop_duplicates(
    subset="lemmatized_text",
    keep="first"
).reset_index(drop=True)

print("Final shape:", df.shape)

Final shape: (5066, 5)


In [384]:
print("Remaining duplicates:", df["lemmatized_text"].duplicated().sum())
print("Empty messages:", (df["lemmatized_text"].str.strip() == "").sum())
print("Missing messages:", df["lemmatized_text"].isnull().sum())

Remaining duplicates: 0
Empty messages: 0
Missing messages: 0


Re-cheking the distribution after cleaning.


In [385]:
distribution = pd.DataFrame({
    "Count": df["label"].value_counts(),
    "Percentage": (
        df["label"].value_counts(normalize=True) * 100
    ).round(2)
})


ham_count = (df["label"] == "ham").sum()
spam_count = (df["label"] == "spam").sum()

ratio = ham_count / spam_count

print("Ham messages:", ham_count)
print("Spam messages:", spam_count)
print("Ham-to-spam ratio:", round(ratio, 2))

Ham messages: 4482
Spam messages: 584
Ham-to-spam ratio: 7.67


As we can see the dataset is imbalanced even after cleaning, with approximately 7 ham messages for every 1 spam
To avoid creating an unrealistic evaluation set, the dataset will first be divided into training and testing sets. This preserves the original proportion of spam and ham messages in both sets.

Random undersampling will then be applied only to the training set by reducing the number of ham messages. The test set will remain unchanged so that the final evaluation reflects the original dataset distribution.

In [386]:
from sklearn.model_selection import train_test_split

# 70% training and 30% temporary data
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["labeled"]
)

#deviding the temp data equally, 15% testing and 15% validation.
validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["labeled"]
)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (3546, 5)
Validation shape: (760, 5)
Testing shape: (760, 5)


In [387]:
import pandas as pd

# seperating the ham and spam messages from the training dataset only
train_ham = train_df[train_df["label"] == "ham"]
train_spam = train_df[train_df["label"] == "spam"]

print("Before downsampling:")
print(train_df["label"].value_counts())

#we will reduce the ham-to-spam ratio to 2, 2 ham messages to every 1 spam message.
target_ham_count = 2 * len(train_spam)

train_ham_downsampled = train_ham.sample(
    n=target_ham_count,
    random_state=42
)

#combine the spam and ham then shuffle the balanced training data
train_balanced = pd.concat(
    [train_ham_downsampled, train_spam],
    ignore_index=True
)

train_balanced = train_balanced.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nAfter downsampling:")
print(train_balanced["label"].value_counts())

Before downsampling:
label
ham     3137
spam     409
Name: count, dtype: int64

After downsampling:
label
ham     818
spam    409
Name: count, dtype: int64


In [388]:
import os

os.makedirs("../data", exist_ok=True)

# Original training split before downsampling
train_df.to_csv("../data/train.csv", index=False)

# Downsampled training set used for model training
train_balanced.to_csv("../data/train_balanced.csv", index=False)

# Validation and test sets
validation_df.to_csv("../data/validation.csv", index=False)
test_df.to_csv("../data/test.csv", index=False)

print("Datasets saved successfully.")

Datasets saved successfully.


To ensure that the testing dataset and the validation both have the same distribution as the main dataset we will print the new distribution of the 3 datasets

In [389]:
print("Training distribution after downsampling:")
print(train_balanced["label"].value_counts())

print("\nValidation distribution unchanged:")
print(validation_df["label"].value_counts())

print("\nTest distribution unchanged:")
print(test_df["label"].value_counts())

Training distribution after downsampling:
label
ham     818
spam    409
Name: count, dtype: int64

Validation distribution unchanged:
label
ham     673
spam     87
Name: count, dtype: int64

Test distribution unchanged:
label
ham     672
spam     88
Name: count, dtype: int64


In [390]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [391]:
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [392]:
TEXT_COLUMN = "lemmatized_text"
TARGET_COLUMN = "labeled"

X_train_text = (
    train_balanced[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .tolist()
)

X_val_text = (
    validation_df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .tolist()
)

X_test_text = (
    test_df[TEXT_COLUMN]
    .fillna("")
    .astype(str)
    .tolist() #converts pandas series to regular python list
)

y_train_array = (
    train_balanced[TARGET_COLUMN]
    .astype(np.float32)
    .to_numpy()
)

y_val_array = (
    validation_df[TARGET_COLUMN]
    .astype(np.float32)
    .to_numpy()
)

y_test_array = (
    test_df[TARGET_COLUMN]
    .astype(np.float32)
    .to_numpy()
)

print("Training messages:", len(X_train_text))
print("Validation messages:", len(X_val_text))
print("Testing messages:", len(X_test_text))

Training messages: 1227
Validation messages: 760
Testing messages: 760


In [393]:


tfidf_vectorizer = TfidfVectorizer(
   
    ngram_range=(1, 2), #unigrams and bigrams are extracted
    max_df=0.95, #if a feature appears more than 95% it is not informative we can remove it
    
)

X_train_tfidf = tfidf_vectorizer.fit_transform(
    X_train_text
)

X_val_tfidf = tfidf_vectorizer.transform(
    X_val_text
)

X_test_tfidf = tfidf_vectorizer.transform(
    X_test_text
)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (1227, 11492)
Validation TF-IDF shape: (760, 11492)
Testing TF-IDF shape: (760, 11492)


In [394]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Number of TF-IDF features:", len(feature_names))
print("First 30 features:")
print(feature_names[:30])

Number of TF-IDF features: 11492
First 30 features:
['aa' 'aa exhaust' 'ab' 'ab numbertoken' 'abel' 'abiola' 'abj' 'abj serve'
 'able' 'able come' 'able give' 'able pay' 'abnormally' 'abnormally call'
 'absolutly' 'absolutly fine' 'abt' 'abt dvg' 'abt leona' 'abt make'
 'abt mei' 'abt muz' 'abt tat' 'abta' 'abta complimentary' 'aburo'
 'aburo enjoy' 'ac' 'ac jsco' 'ac post']


In [395]:
import torch

X_train_tensor = torch.tensor(
    X_train_tfidf.toarray(),
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_tfidf.toarray(),
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_tfidf.toarray(),
    dtype=torch.float32
)

In [396]:
y_train_tensor = torch.tensor(
    y_train_array,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val_array,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test_array,
    dtype=torch.float32
)

In [397]:
print("X train:", X_train_tensor.shape)
print("y train:", y_train_tensor.shape)

print("X validation:", X_val_tensor.shape)
print("y validation:", y_val_tensor.shape)

print("X test:", X_test_tensor.shape)
print("y test:", y_test_tensor.shape)

X train: torch.Size([1227, 11492])
y train: torch.Size([1227])
X validation: torch.Size([760, 11492])
y validation: torch.Size([760])
X test: torch.Size([760, 11492])
y test: torch.Size([760])
